# How Rare Is Genitive _-ss_?

Menota consists of 91 transcriptions, 86 of which have a diplomatic level of transcription while 70 have been lemmatized. We'll use the absence of final _s_ in the lemma form as a rough proxy for words we don't expect to have forms ending in _-ss_, and then we can perform a count of such lemmas ending in _-ss_ in the diplomatic output.

As geminates are often indicated in the manuscripts with small capitals or dotted letters, and some Menota transcriptions maintain this convention on the diplomatic level, we will want to include those in our count.

## Preprocessing

In [99]:
import os,glob,re
from lxml import etree
from collections import Counter

In [87]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)

menota = dict()
for doc in glob.glob('menota/noent/*xml'):
    title = os.path.basename(doc)[:-4]
    tree = etree.parse(doc, parser=parser)
    root = tree.getroot()
    tokens = [w for w in root.iter('{http://www.tei-c.org/ns/1.0}w')]
    menota[title] = tokens


In [88]:
menota_dipl_lemmatized = dict()
for k,v in menota.items():
    if len(v[0].findall('.//{http://www.menota.org/ns/1.0}dipl')) > 0 and v[0].get('lemma') is not None:
        menota_dipl_lemmatized[k] = v

In [89]:
len(menota_dipl_lemmatized.items())

61

In [90]:
type(menota_dipl_lemmatized['am132_kormaks_saga'][0])

lxml.etree._Element

We are now at 61 Menota documents that have both been lemmatized and transcribed at a diplomatic level. We won't have to account for `dg4-7` documents separately (contrast `menota-extract.ipynb`) because those of its members that do not use Menota's `<dipl>` element have not been lemmatized.

## Tally

In [ ]:
data = dict()
for title,tokens in menota_dipl_lemmatized.items():
    tally = dict()
    for token in tokens:
        lemma = token.get('lemma')
        if lemma is not None and lemma not in ['sá', 'sjá', 'vér', 'hverr', 'einnhverr', 'engi', 'várr', 'annarr', 'yðarr', 'hvarr', 'nǫkkurr']:
            if len(lemma) > 0:
                if lemma[-1] != 's':
                    if len(token.findall('.//{http://www.menota.org/ns/1.0}dipl')) > 0:
                        form = etree.tostring(token.find('.//{http://www.menota.org/ns/1.0}dipl'), encoding='unicode', method='text')
                        if len(form) > 2:
                            if re.search("[sſꜱ][sſꜱ]$", form):
                                if lemma in tally.keys():
                                    tally[lemma] = tally[lemma] + 1
                                else:
                                    tally[lemma] = 1
    data[title] = tally

In [92]:
data['holmPerg17_thomass_saga']

{'eptirdœmi': 3,
 'herbergi': 11,
 'ríki': 2,
 'léttir': 1,
 'efni': 3,
 'sundrlyndi': 5,
 'lítillæti': 1,
 'lesa': 3,
 'réttlæti': 4,
 'sæti': 6,
 'umskipti': 1,
 'erendi': 2,
 'kiósa': 4,
 'rísa': 4,
 'vandlæti': 1,
 'klæði': 1,
 'samþykki': 1,
 'iarðríki': 1,
 'hvárgi': 1,
 'siá': 1,
 'Frankaríki': 2,
 'eptirlæti': 1,
 'hirðir': 1,
 'embætti': 1,
 'hásæti': 1,
 'enni': 1,
 'Kantarabyrgi': 1}

In [93]:
len(data)

61

In [94]:
data['am132_egils_saga']

{'bú': 14,
 'ríki': 6,
 'sǽti': 2,
 'heimfararleyfi': 1,
 'fjǫlmenni': 2,
 'skáldaspillir': 1,
 'hildir': 1,
 'ørendi': 2,
 'yngvarr': 1,
 'útbú': 1,
 'viðskifti': 1,
 'þórir': 7,
 'ólífi': 1,
 'hersir': 3,
 'hlutskifti': 1,
 'skifti': 1,
 'samþykki': 1,
 'alþingi': 1,
 'gellir': 1,
 'smámenni': 1}

In [102]:
data['am132_njals_saga']

{'hersir': 3,
 'garðaríki': 1,
 'bú': 4,
 'einvígi': 2,
 'tvímǽli': 2,
 'holtaþórir': 3,
 'eftirlǽti': 1,
 'alþingi': 10,
 'gunnarr': 2,
 'ágǽti': 2,
 'ríki': 1,
 'sǽti': 1,
 'ólífi': 1,
 'þórir': 5,
 'geitir': 1,
 'fellshverfi': 1,
 'skógahverfi': 2,
 'hildir': 4,
 'liðsinni': 3,
 'seyðir': 1,
 'fagrmǽli': 1,
 'vígi': 2,
 'vǽtti': 1,
 'jafnsǽtti': 1,
 'árǽði': 2,
 'heimili': 1,
 'leyni': 1}

In [119]:
tallies_per_doc = dict()
for k,v in data.items():
    counts = sum(i for i in v.values())
    tallies_per_doc[k] = counts
rankings = Counter(tallies_per_doc).most_common()
normalized_tallies_per_doc = {}
for k,v in rankings:
    normalized_frequency = (round(v/len(menota[k]) * 1000, 2))
    normalized_tallies_per_doc[k] = normalized_frequency
normalized_rankings = Counter(normalized_tallies_per_doc).most_common()
combined_rankings = []
for k,v in normalized_rankings:
    this_ranking = (k, tallies_per_doc[k], v, len(menota[k]))
    combined_rankings.append(this_ranking)

In [121]:
for title, score, normalized, length in combined_rankings:
    print(f"{title} {score} {normalized} {length}")

meII2_frostathingslog 4 1.68 2388
am132_droplaugasona_saga 12 1.24 9692
holmPerg17_thomass_saga 64 1.05 61087
am132_viga-glums_saga 20 1.03 19478
lbsFragm82_olafs_saga_helga 2 1.02 1969
am132_bandamanna_saga 11 0.98 11275
nraNorrFragm59_rimbegla 1 0.9 1117
holmPerg6_barlaams_saga 66 0.86 76412
am132_egils_saga 49 0.82 59877
wolfAug9-10_egils_saga 29 0.75 38618
am113b_islendingabok 3 0.74 4035
am132_njals_saga 59 0.72 82259
am132_olkofra_thattr 2 0.69 2918
am178_thidreks_saga 17 0.66 25673
nraNorrFragm63_karlamagnuss_saga 1 0.58 1714
am132_hallfredar_saga 5 0.51 9810
am132_finnboga_saga 9 0.38 23607
am132_laxdoela_saga 21 0.34 61897
am132_kormaks_saga 3 0.23 12876
holmPerg34_landslog 8 0.18 45243
am132_fostbraedra_saga 1 0.18 5703
am677_gregory 7 0.14 51727
dg4-7_strengleikar 4 0.1 38453
holmPerg4_thidreks_saga 10 0.09 115205
am519a_alexanders_saga 3 0.06 46797
am619_norwegian_homily_book 1 0.02 60117
nraNorrFragm80_pals_saga 0 0.0 1752
dg8II_olafs_saga 0 0.0 41142
gks2365_voluspa 0 0.0

In [122]:
data['meII2_frostathingslog']

{'sjá': 4}